# Feature Engineering

Tier A/B feature builders (Decision #1), target definition/transforms (Decisions #2, #3), failed-job exclusion (Decision #4), embedding dimensionality reduction (Decision #7).

See the conceptualization plan and EXPERIMENT_TRACKER.md for full context.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src import config, features, metrics, plotting, models, roofline


## Load data

Same `SAMPLE_SIZE` dev-sample pattern as notebook 01 — see `EXPERIMENT_TRACKER.md` for when this needs to step up toward the full dataset.

In [ ]:
FDATA_DIR = "../data/raw/fdata"
PM100_PATH = "../data/raw/pm100/pm100_job_table.parquet"
SAMPLE_SIZE = 5000

import glob
fdata_files = sorted(glob.glob(f"{FDATA_DIR}/*.parquet"))
fdata = pd.concat([pd.read_parquet(f) for f in fdata_files], ignore_index=True)
pm100 = pd.read_parquet(PM100_PATH)

if SAMPLE_SIZE is not None:
    fdata = fdata.sample(n=min(SAMPLE_SIZE, len(fdata)), random_state=0).reset_index(drop=True)
    pm100 = pm100.sample(n=min(SAMPLE_SIZE, len(pm100)), random_state=0).reset_index(drop=True)

FDATA_DATETIME_COLS = ["adt", "qdt", "schedsdt", "deldt", "sdt", "edt"]
for c in FDATA_DATETIME_COLS:
    fdata[c] = pd.to_datetime(fdata[c], errors="coerce")
pm100["submit_time"] = pd.to_datetime(pm100["submit_time"], unit="s", errors="coerce")
pm100["start_time"] = pd.to_datetime(pm100["start_time"], unit="s", errors="coerce")
pm100["end_time"] = pd.to_datetime(pm100["end_time"], unit="s", errors="coerce")

print("fdata:", fdata.shape, " pm100:", pm100.shape)


## Step 1: Exclude failed/cancelled/killed jobs (Decision #4)

Their duration/power are artifacts of failure, not real workload behavior. F-DATA's `exit state` is a clean completed/failed binary; PM100's `job_state` has 6 values, only `COMPLETED` reflects normal execution.

In [ ]:
print("F-DATA exit state breakdown:")
print(fdata["exit state"].value_counts())
print("\nPM100 job_state breakdown:")
print(pm100["job_state"].value_counts())

fdata = features.filter_completed_jobs(fdata, "fdata")
pm100 = features.filter_completed_jobs(pm100, "pm100")


## Step 2: Tier A/B feature matrices (Decision #1)

Re-running the leakage sanity check from notebook 01 on the post-exclusion data — this should still pass, but re-verifying after every transformation step is the point of Decision #19.

In [ ]:
fdata_tier_a = features.build_tier_a_features(fdata, "fdata")
fdata_tier_b = features.build_tier_b_features(fdata, "fdata")
pm100_tier_a = features.build_tier_a_features(pm100, "pm100")
pm100_tier_b = features.build_tier_b_features(pm100, "pm100")

for name, cols, tier, dataset in [
    ("F-DATA Tier A", fdata_tier_a.columns, "A", "fdata"),
    ("F-DATA Tier B", fdata_tier_b.columns, "B", "fdata"),
    ("PM100 Tier A", pm100_tier_a.columns, "A", "pm100"),
    ("PM100 Tier B", pm100_tier_b.columns, "B", "pm100"),
]:
    features.assert_no_tier_leakage(list(cols), tier, dataset)
    print(f"{name}: {len(cols)} columns, OK")


## Step 3: Embedding dimensionality reduction (Decision #7)

F-DATA only — PCA-reduce the 384-dim SBert `embedding` column to 10 components for the tree-based models. PM100 has no job-name embedding field.

In [ ]:
fdata_emb_pca = features.reduce_fdata_embedding(fdata, n_components=10)
print(fdata_emb_pca.shape)
fdata_emb_pca.describe().T


## Step 4: Historical rolling-stat features (Tier A)

Each user's mean `duration` over their *previous* jobs only (never the current job's own outcome — computed via `shift(1)` before the rolling window, per Decision #1's leakage discipline). With only 1 of F-DATA's 38 months present and a 5000-row dev sample, expect these to be sparse/noisy — this is expected, not a bug (see EXPERIMENT_TRACKER.md).

In [ ]:
fdata_with_rolling = features.add_user_rolling_stat(fdata, "fdata", "duration", window=5)
pm100_with_rolling = features.add_user_rolling_stat(pm100, "pm100", "run_time", window=5)

n_fdata_available = fdata_with_rolling["duration_user_rolling_mean"].notna().sum()
n_pm100_available = pm100_with_rolling["run_time_user_rolling_mean"].notna().sum()
print(f"F-DATA: rolling stat available for {n_fdata_available}/{len(fdata_with_rolling)} jobs")
print(f"PM100: rolling stat available for {n_pm100_available}/{len(pm100_with_rolling)} jobs")

fdata_with_rolling[["usr", "adt", "duration", "duration_user_rolling_mean"]].head(10)


## Step 5: Target transforms (Decision #3)

log1p on all three targets per dataset (execution time, memory, power) — training happens in log-space, metrics get reported in both log-space and back-transformed real units. Sanity-checking the round-trip here (Decision #19) rather than assuming it.

In [ ]:
for name, targets, df in [("F-DATA", features.FDATA_TARGETS, fdata), ("PM100", features.PM100_TARGETS, pm100)]:
    for target_name, col in targets.items():
        raw = df[col].dropna().to_numpy()
        if len(raw) and hasattr(raw[0], "__len__"):
            # PM100 power: array-valued, reduce to per-job mean first (see notebook 01)
            raw = np.array([np.mean(a) for a in raw])
        raw = raw[raw >= 0].astype(float)
        transformed = features.transform_target(raw)
        recovered = features.inverse_transform_target(transformed)
        metrics.expm1_round_trip_check(raw)
        print(f"{name} {target_name} ({col}): round-trip OK, n={len(raw)}, "
              f"raw range [{raw.min():.2f}, {raw.max():.2f}], "
              f"log1p range [{transformed.min():.2f}, {transformed.max():.2f}]")


## Summary

This notebook validates the feature-engineering pipeline end to end on the dev sample: exclusion, Tier A/B separation, embedding reduction, rolling-stat features, and target transforms all pass their sanity checks. Nothing here is yet the "final" feature matrix used for training — that assembly happens in notebooks 05-07, reusing these same `src/features.py` functions against the full dataset once available.